In [1]:
import pandas as pd
import os
# from rapidfuzz import process, fuzz
import unicodedata
import requests
import csv
import difflib
import html

### Util functions


In [16]:
haaland['kickoff_time']

0     2024-08-18T15:30:00Z
1     2024-08-24T14:00:00Z
2     2024-08-31T16:30:00Z
3     2024-09-14T14:00:00Z
4     2024-09-22T15:30:00Z
5     2024-09-28T11:30:00Z
6     2024-10-05T14:00:00Z
7     2024-10-20T13:00:00Z
8     2024-10-26T14:00:00Z
9     2024-11-02T15:00:00Z
10    2024-11-09T17:30:00Z
11    2024-11-23T17:30:00Z
12    2024-12-01T16:00:00Z
13    2024-12-04T19:30:00Z
14    2024-12-07T15:00:00Z
15    2024-12-15T16:30:00Z
16    2024-12-21T12:30:00Z
17    2024-12-26T12:30:00Z
18    2024-12-29T14:30:00Z
19    2025-01-04T15:00:00Z
20    2025-01-14T19:30:00Z
21    2025-01-19T16:30:00Z
22    2025-01-25T17:30:00Z
23    2025-02-02T16:30:00Z
24    2025-02-15T15:00:00Z
25    2025-02-23T16:30:00Z
26    2025-02-26T19:30:00Z
27    2025-03-08T12:30:00Z
28    2025-03-15T15:00:00Z
29    2025-04-02T18:45:00Z
30    2025-04-06T15:30:00Z
31    2025-04-12T11:30:00Z
32    2025-04-19T14:00:00Z
34    2025-05-02T19:00:00Z
35    2025-05-10T14:00:00Z
36    2025-05-20T19:00:00Z
37    2025-05-25T15:00:00Z
N

In [29]:
haaland = pd.read_csv('./data/joint/24-25/fpl/Cole Palmer.csv')
haaland['date'] = haaland['kickoff_time'].str.split('T').str[0]
haaland[['date', 'kickoff_time', 'round']]

,date,kickoff_time,round
0,2024-08-18,2024-08-18T15:30:00Z,1
1,2024-08-25,2024-08-25T13:00:00Z,2
2,2024-09-01,2024-09-01T12:30:00Z,3
3,2024-09-14,2024-09-14T19:00:00Z,4
4,2024-09-21,2024-09-21T11:30:00Z,5
5,2024-09-28,2024-09-28T14:00:00Z,6
6,2024-10-06,2024-10-06T13:00:00Z,7
7,2024-10-20,2024-10-20T15:30:00Z,8
8,2024-10-27,2024-10-27T14:00:00Z,9
9,2024-11-03,2024-11-03T16:30:00Z,10


In [30]:
haaland_under = pd.read_csv('./data/joint/24-25/understat/Cole Palmer.csv')
haaland_under[haaland_under['season'] == 2024][['h_team', 'a_team', 'date', 'season']]

,h_team,a_team,date,season
3,Nottingham Forest,Chelsea,2025-05-25,2024
4,Chelsea,Manchester United,2025-05-16,2024
5,Newcastle United,Chelsea,2025-05-11,2024
6,Chelsea,Liverpool,2025-05-04,2024
7,Chelsea,Everton,2025-04-26,2024
8,Fulham,Chelsea,2025-04-20,2024
9,Chelsea,Ipswich,2025-04-13,2024
10,Brentford,Chelsea,2025-04-06,2024
11,Chelsea,Tottenham,2025-04-03,2024
12,Chelsea,Leicester,2025-03-09,2024


In [31]:
pd.merge(haaland, haaland_under, on=['date'])[['round', 'h_team', 'a_team', 'date', 'season']]

,round,h_team,a_team,date,season
0,1,Chelsea,Manchester City,2024-08-18,2024
1,2,Wolverhampton Wanderers,Chelsea,2024-08-25,2024
2,3,Chelsea,Crystal Palace,2024-09-01,2024
3,4,Bournemouth,Chelsea,2024-09-14,2024
4,5,West Ham,Chelsea,2024-09-21,2024
5,6,Chelsea,Brighton,2024-09-28,2024
6,7,Chelsea,Nottingham Forest,2024-10-06,2024
7,8,Liverpool,Chelsea,2024-10-20,2024
8,9,Chelsea,Newcastle United,2024-10-27,2024
9,10,Manchester United,Chelsea,2024-11-03,2024


In [72]:
def map_names(sns):
    print('Mapping names for season: ', sns)
    fpl_raw = next(os.walk(f"./data/joint/{sns}/fpl/"), (None, None, []))[2]
    understat_raw = next(os.walk(f"./data/joint/{sns}/understat/"), (None, None, []))[2]

    fpl_names = [name.split('.')[0] for name in fpl_raw]
    understat_names = [name.split('.')[0] for name in understat_raw]

    # 2. Logic for cleaning and mapping
    def normalize_name(name):
        """
        Removes .csv, lowers case, removes accents, removes non-alphanumeric
        to make matching easier.
        """
        if not name: return ""

        # Remove extension
        name = name.replace('.csv', '').lower()

        # Remove accents (e.g., é -> e, ö -> o)
        name = ''.join(c for c in unicodedata.normalize('NFD', name)
                    if unicodedata.category(c) != 'Mn')

        return name+'.csv'

    # 3. MANUAL OVERRIDES
    # These are specific mappings where the string similarity is too low
    # or ambiguous for automatic detection.
    manual_overrides = {
        'Alisson Ramses Becker.csv': 'Alisson.csv',
        'Amad_Diallo.csv': 'Amad_Diallo_Traore_8127.csv',  # Specific FPL naming
        'Antony Matheus dos Santos.csv': 'Antony.csv',
        'Ben Brereton.csv': 'Ben Brereton Díaz.csv',
        'Benoît Badiashile.csv': 'Benoit Badiashile Mukinayi.csv',
        'Bernardo Veiga de Carvalho e Silva.csv': 'Bernardo Silva.csv',
        'Bobby De Cordova-Reid.csv': 'Bobby Reid.csv',
        'Bruno Borges Fernandes.csv': 'Bruno Fernandes.csv',
        'Bruno Guimarães Rodriguez Moura.csv': 'Bruno Guimarães.csv',
        'Carlos Henrique Casimiro.csv': 'Casemiro.csv',
        'Carlos Vinícius Alves Morais.csv': 'Carlos Vinicius.csv',
        'Cédric Alves Soares.csv': 'Cédric Soares.csv',
        'Daniel Castelo Podence.csv': 'Daniel Podence.csv',
        'Danilo dos Santos de Oliveira.csv': 'Danilo.csv',
        'Darwin Núñez Ribeiro.csv': 'Darwin Núñez.csv',
        'David Raya Martin.csv': 'David Raya.csv',
        'Deivid Washington de Souza Eugênio.csv': 'Deivid Washington.csv',
        'Diego Carlos Santos Silva.csv': 'Diego Carlos.csv',
        'Diogo Dalot Teixeira.csv': 'Diogo Dalot.csv',
        'Diogo Teixeira da Silva.csv': 'Diogo Jota.csv', # Completely different
        'Dominic Dos Santos Martins.csv': 'Dominic Solanke.csv', # Wait, this might be error in FPL data or specific alias? Assume Solanke based on list context or specific player
        # Actually, Solanke is in the list as Solanke. "Dos Santos Martins" is usually Dom (Solanke).

        'Douglas Luiz Soares de Paulo.csv': 'Douglas Luiz.csv',
        'Ederson Santana de Moraes.csv': 'Ederson.csv',
        'Edson Álvarez Velázquez.csv': 'Edson Álvarez.csv',
        'Emerson Leite de Souza Junior.csv': 'Emerson.csv',
        'Emerson Palmieri dos Santos.csv': 'Emerson.csv', # Be careful with duplicates here
        'Emiliano Buendía Stati.csv': 'Emiliano Buendía.csv',
        'Emiliano Martínez Romero.csv': 'Emiliano Martinez.csv',
        'Fabio Henrique Tavares.csv': 'Fabinho.csv',
        'Felipe Augusto de Almeida Monteiro.csv': 'Felipe.csv',
        'Frederico Rodrigues de Paula Santos.csv': 'Fred.csv',
        'Fábio Ferreira Vieira.csv': 'Fábio Vieira.csv',
        'Gabriel dos Santos Magalhães.csv': 'Gabriel.csv',
        'Gabriel Fernando de Jesus.csv': 'Gabriel Jesus.csv',
        'Gabriel Martinelli Silva.csv': 'Gabriel Martinelli.csv',
        'Gonçalo Manuel Ganchinho Guedes.csv': 'Gonçalo Guedes.csv',
        'Gustavo Henrique Furtado Scarpa.csv': 'Gustavo Scarpa.csv',
        'Hwang Hee-chan.csv': 'Hee-Chan Hwang.csv',
        'Igor Julio dos Santos de Paulo.csv': 'Igor Julio.csv',
        'Ivan Neves Abreu Cavaleiro.csv': 'Ivan Cavaleiro.csv', # Not in understat list?
        'Jaden Philogene-Bidace.csv': 'Jaden Philogene-Bidace.csv', # exact match
        'Jefferson Lerma Solís.csv': 'Jefferson Lerma.csv',
        'Joelinton Cássio Apolinário de Lira.csv': 'Joelinton.csv',
        'Jorge Luiz Frello Filho.csv': 'Jorginho.csv',
        'José Malheiro de Sá.csv': 'José Sá.csv',
        'João Pedro Junqueira de Jesus.csv': 'João Pedro.csv',
        'João Victor Gomes da Silva.csv': 'João Gomes.csv',
        'Kepa Arrizabalaga.csv': 'Kepa.csv',
        'Lucas Tolentino Coelho de Lima.csv': 'Lucas Paquetá.csv',
        'Manuel Benson Hedilazio.csv': 'Benson Manuel.csv', # Reversed name
        'Marc Cucurella Saseta.csv': 'Marc Cucurella.csv',
        'Martin Ødegaard.csv': 'Martin Odegaard.csv',
        'Matheus França de Oliveira.csv': 'Matheus França.csv',
        'Matheus Luiz Nunes.csv': 'Matheus Nunes.csv',
        'Matheus Santos Carneiro Da Cunha.csv': 'Matheus Cunha.csv',
        'Miguel Almirón Rejala.csv': 'Miguel Almirón.csv',
        'Murillo Santiago Costa dos Santos.csv': 'Murillo.csv',
        'Norberto Bercique Gomes Betuncal.csv': 'Beto.csv',
        'Norberto Murara Neto.csv': 'Neto.csv',
        'Pedro Lomba Neto.csv': 'Pedro Neto.csv',
        'Philippe Coutinho Correia.csv': 'Philippe Coutinho.csv',
        'Richarlison de Andrade.csv': 'Richarlison.csv',
        'Rodrigo Hernandez.csv': 'Rodri.csv',
        'Romelu Lukaku Bolingoli.csv': 'Romelu Lukaku.csv', # Not in list? Check list
        'Rúben Gato Alves Dias.csv': 'Rúben Dias.csv',
        'Tanguy Ndombélé Alvaro.csv': 'Tanguy Ndombele.csv',
        'Thiago Alcántara do Nascimento.csv': 'Thiago Alcántara.csv',
        'Thiago Emiliano da Silva.csv': 'Thiago Silva.csv',
        'Tino Livramento.csv': 'Valentino Livramento.csv',
        'Tomáš Souček.csv': 'Tomas Soucek.csv',
        'Toti António Gomes.csv': 'Toti.csv',
        'Vini de Souza Costa.csv': 'Vinicius Souza.csv',
        'Wesley Moraes Ferreira da Silva.csv': 'Wesley.csv', # Check list
        'Willian Borges da Silva.csv': 'Willian.csv',
        'Đorđe Petrović.csv': 'Djordje Petrovic.csv',
        'Amari\'i Bell.csv': 'Amari&#039;i Bell.csv'
    }

    #

    final_mapping = {}
    unmapped = []

    # Set up simplified dictionaries for lookup
    understat_simple_map = {normalize_name(x): x for x in understat_raw}

    for fpl_file in fpl_raw:
        # 1. Check Manual Override first
        if fpl_file in manual_overrides:
            # Verify the override destination actually exists in understat list to avoid errors
            target = manual_overrides[fpl_file]

            # Check if override target exists in Understat
            if target in understat_raw:
                final_mapping[fpl_file] = target
            else:
                # Fallback: try to normalize the target and find it
                norm_target = normalize_name(target)
                if norm_target in understat_simple_map:
                    final_mapping[fpl_file] = understat_simple_map[norm_target]
                else:
                    unmapped.append(fpl_file)
            continue

        # 2. Exact Normalized Match
        fpl_norm = normalize_name(fpl_file)
        if fpl_norm in understat_simple_map:
            final_mapping[fpl_file] = understat_simple_map[fpl_norm]
            continue

        # 3. Fuzzy Match (Original strings)
        matches = difflib.get_close_matches(fpl_file, understat_raw, n=1, cutoff=0.8)
        if matches:
            final_mapping[fpl_file] = matches[0]
            continue

        # 4. Fuzzy Match (Normalized strings)
        norm_matches = difflib.get_close_matches(fpl_norm, list(understat_simple_map.keys()), n=1, cutoff=0.8)
        if norm_matches:
            final_mapping[fpl_file] = understat_simple_map[norm_matches[0]]
            continue

        unmapped.append(fpl_file)


        #     # Some overrides might need slight fuzzy matching if I mistyped the destination above
        #     matches = difflib.get_close_matches(target, understat_names, n=1, cutoff=0.8)

        #     if matches:
        #         final_mapping[fpl_file] = matches[0]
        #     else:
        #         # If exact override not found, try simplified override match
        #         simple_target = normalize_name(target)
        #         if simple_target in understat_simple_map:
        #             final_mapping[fpl_file] = understat_simple_map[simple_target]
        #         else:
        #             print(f"Warning: Manual override target '{target}' for '{fpl_file}' not found in understat names.")

        #     continue

        # # 2. Exact Simplified Match
        # fpl_simple = normalize_name(fpl_file)
        # if fpl_simple in understat_simple_map:
        #     final_mapping[fpl_file] = understat_simple_map[fpl_simple]
        #     continue

        # # 3. Fuzzy Match
        # # Matches based on string similarity (helps with accents, minor spelling diffs)
        # matches = difflib.get_close_matches(fpl_file, understat_names, n=1, cutoff=0.6)
        # if matches:
        #     final_mapping[fpl_file] = matches[0]
        # else:
        #     # 4. Try matching simplified versions fuzzy
        #     matches_simple = difflib.get_close_matches(fpl_simple, list(understat_simple_map.keys()), n=1, cutoff=0.8)
        #     if matches_simple:
        #         final_mapping[fpl_file] = understat_simple_map[matches_simple[0]]
        #     else:
        #         unmapped.append(fpl_file)

    # print('-------------->', final_mapping)
    # Output the Dictionary
    # print("# GENERATED DICTIONARY MAPPING")
    # print("name_map = {")
    # for k, v in sorted(final_mapping.items()):
    #     print(f"    '{k}': '{v}',")
    # print("}")

    print(f"\n# Statistics: Mapped {len(final_mapping)}/{len(fpl_names)}")
    # if unmapped:
    #     print("\n# Unmapped files (Check these manually):")
    #     for u in unmapped:
    #         print(u)

    fpl = final_mapping.keys()
    under = final_mapping.values()

    cleaned_fpl_under_names = pd.DataFrame({'fpl_name': fpl, 'understat_name': under})
    return cleaned_fpl_under_names

In [53]:
# import os
# merged = next(os.walk(f"./data/joint/24-25/merged/"), (None, None, []))[2]
# merged

In [54]:

# fpl = next(os.walk("./data/joint/24-25/fpl/"), (None, None, []))[2]
# fpl

In [55]:
# set(fpl) - set(merged)

In [95]:
# merge_fpl_understat_data("./data/vaastav/data/202/players/", "./data/vaastav/data/2025-26/understat/", "24-25")

In [96]:
# def map_names(sns):
#     """
#     Maps FPL player filenames to Understat player filenames for a given season.
#     Handles encoding differences, common aliases, and manual overrides.
#     """
#     print(f'Mapping names for season: {sns}')

#     # Paths
#     fpl_path = f"./data/joint/{sns}/fpl/"
#     understat_path = f"./data/joint/{sns}/understat/"
#     raw_players_path = f"./data/vaastav/data/202/{sns}/players_raw.csv"

#     # Get file lists
#     fpl_raw = next(os.walk(fpl_path), (None, None, []))[2]
#     understat_raw = next(os.walk(understat_path), (None, None, []))[2]

#     # 1. Normalization Utility
#     def normalize_name(name):
#         """
#         Removes extension, decodes HTML, lowers case, removes accents and special chars.
#         """
#         if not name: return ""
#         # Remove extension
#         name = name.replace('.csv', '')
#         # Decode HTML entities (e.g., &#039; -> ')
#         name = html.unescape(name)
#         # Convert to lowercase
#         name = name.lower()
#         # Remove accents (e.g., é -> e)
#         name = ''.join(c for c in unicodedata.normalize('NFD', name)
#                        if unicodedata.category(c) != 'Mn')
#         # Remove extra whitespace and non-alphanumeric except spaces/hyphens
#         name = "".join([c for c in name if c.isalnum() or c in (" ", "-")])
#         return name.strip()

#     # 2. Manual Overrides for 23-24 Season
#     # These handle players with significantly different names or specific data quirks.
#     manual_overrides_ = {
#         'Alisson Ramses Becker.csv': 'Alisson.csv',
#         'Amad Diallo.csv': 'Amad Diallo Traore.csv',
#         'Antony Matheus dos Santos.csv': 'Antony.csv',
#         'Ben Brereton.csv': 'Ben Brereton Díaz.csv',
#         'Benoît Badiashile.csv': 'Benoit Badiashile Mukinayi.csv',
#         'Bernardo Veiga de Carvalho e Silva.csv': 'Bernardo Silva.csv',
#         'Bobby De Cordova-Reid.csv': 'Bobby Reid.csv',
#         'Bruno Borges Fernandes.csv': 'Bruno Fernandes.csv',
#         'Bruno Guimarães Rodriguez Moura.csv': 'Bruno Guimarães.csv',
#         'Carlos Henrique Casimiro.csv': 'Casemiro.csv',
#         'Carlos Vinícius Alves Morais.csv': 'Carlos Vinicius.csv',
#         'Cédric Alves Soares.csv': 'Cédric Soares.csv',
#         'Daniel Castelo Podence.csv': 'Daniel Podence.csv',
#         'Danilo dos Santos de Oliveira.csv': 'Danilo.csv',
#         'Darwin Núñez Ribeiro.csv': 'Darwin Núñez.csv',
#         'David Raya Martin.csv': 'David Raya.csv',
#         'Deivid Washington de Souza Eugênio.csv': 'Deivid Washington.csv',
#         'Diego Carlos Santos Silva.csv': 'Diego Carlos.csv',
#         'Diogo Dalot Teixeira.csv': 'Diogo Dalot.csv',
#         'Diogo Teixeira da Silva.csv': 'Diogo Jota.csv',
#         'Dominic Dos Santos Martins.csv': 'Dominic Solanke.csv',
#         'Douglas Luiz Soares de Paulo.csv': 'Douglas Luiz.csv',
#         'Ederson Santana de Moraes.csv': 'Ederson.csv',
#         'Edson Álvarez Velázquez.csv': 'Edson Álvarez.csv',
#         'Emerson Leite de Souza Junior.csv': 'Emerson.csv',
#         'Emerson Palmieri dos Santos.csv': 'Emerson.csv',
#         'Emiliano Buendía Stati.csv': 'Emiliano Buendía.csv',
#         'Emiliano Martínez Romero.csv': 'Emiliano Martinez.csv',
#         'Fabio Henrique Tavares.csv': 'Fabinho.csv',
#         'Felipe Augusto de Almeida Monteiro.csv': 'Felipe.csv',
#         'Frederico Rodrigues de Paula Santos.csv': 'Fred.csv',
#         'Fábio Ferreira Vieira.csv': 'Fábio Vieira.csv',
#         'Gabriel dos Santos Magalhães.csv': 'Gabriel.csv',
#         'Gabriel Fernando de Jesus.csv': 'Gabriel Jesus.csv',
#         'Gabriel Martinelli Silva.csv': 'Gabriel Martinelli.csv',
#         'Gonçalo Manuel Ganchinho Guedes.csv': 'Gonçalo Guedes.csv',
#         'Gustavo Henrique Furtado Scarpa.csv': 'Gustavo Scarpa.csv',
#         'Hwang Hee-chan.csv': 'Hee-Chan Hwang.csv',
#         'Igor Julio dos Santos de Paulo.csv': 'Igor Julio.csv',
#         'Jefferson Lerma Solís.csv': 'Jefferson Lerma.csv',
#         'Joelinton Cássio Apolinário de Lira.csv': 'Joelinton.csv',
#         'Jorge Luiz Frello Filho.csv': 'Jorginho.csv',
#         'José Malheiro de Sá.csv': 'José Sá.csv',
#         'João Pedro Junqueira de Jesus.csv': 'João Pedro.csv',
#         'João Victor Gomes da Silva.csv': 'João Gomes.csv',
#         'João Palhinha Gonçalves.csv': 'João Palhinha.csv',
#         'Kepa Arrizabalaga.csv': 'Kepa.csv',
#         'Lucas Tolentino Coelho de Lima.csv': 'Lucas Paquetá.csv',
#         'Manuel Benson Hedilazio.csv': 'Benson Manuel.csv',
#         'Marc Cucurella Saseta.csv': 'Marc Cucurella.csv',
#         'Martin Ødegaard.csv': 'Martin Odegaard.csv',
#         'Matheus França de Oliveira.csv': 'Matheus França.csv',
#         'Matheus Luiz Nunes.csv': 'Matheus Nunes.csv',
#         'Matheus Santos Carneiro Da Cunha.csv': 'Matheus Cunha.csv',
#         'Miguel Almirón Rejala.csv': 'Miguel Almirón.csv',
#         'Murillo Santiago Costa dos Santos.csv': 'Murillo.csv',
#         'Norberto Bercique Gomes Betuncal.csv': 'Beto.csv',
#         'Norberto Murara Neto.csv': 'Neto.csv',
#         'Pedro Lomba Neto.csv': 'Pedro Neto.csv',
#         'Philippe Coutinho Correia.csv': 'Philippe Coutinho.csv',
#         'Richarlison de Andrade.csv': 'Richarlison.csv',
#         'Rodrigo Hernandez.csv': 'Rodri.csv',
#         'Rúben Gato Alves Dias.csv': 'Rúben Dias.csv',
#         'Son Heung-min.csv': 'Son Heung-Min.csv',
#         'Thiago Alcántara do Nascimento.csv': 'Thiago Alcántara.csv',
#         'Thiago Emiliano da Silva.csv': 'Thiago Silva.csv',
#         'Tino Livramento.csv': 'Valentino Livramento.csv',
#         'Tomáš Souček.csv': 'Tomas Soucek.csv',
#         'Toti António Gomes.csv': 'Toti.csv',
#         'Vini de Souza Costa.csv': 'Vinicius Souza.csv',
#         'Willian Borges da Silva.csv': 'Willian.csv',
#         'Đorđe Petrović.csv': 'Djordje Petrovic.csv',
#         'Pervis Estupiñán.csv': 'Estupiñán.csv'
#     }

#     manual_overrides = {
#     # --- Classic/Legacy Overrides (Pre-24/25) ---
#     'Alisson Ramses Becker.csv': 'Alisson.csv',
#     'Amad Diallo.csv': 'Amad Diallo Traore.csv',
#     'Antony Matheus dos Santos.csv': 'Antony.csv',
#     'Benoît Badiashile.csv': 'Benoit Badiashile Mukinayi.csv',
#     'Bernardo Veiga de Carvalho e Silva.csv': 'Bernardo Silva.csv',
#     'Bobby De Cordova-Reid.csv': 'Bobby Reid.csv',
#     'Bruno Borges Fernandes.csv': 'Bruno Fernandes.csv',
#     'Bruno Guimarães Rodriguez Moura.csv': 'Bruno Guimarães.csv',
#     'Carlos Henrique Casimiro.csv': 'Casemiro.csv',
#     'Carlos Vinícius Alves Morais.csv': 'Carlos Vinicius.csv',
#     'Cédric Alves Soares.csv': 'Cédric Soares.csv',
#     'Daniel Castelo Podence.csv': 'Daniel Podence.csv',
#     'Danilo dos Santos de Oliveira.csv': 'Danilo.csv',
#     'Darwin Núñez Ribeiro.csv': 'Darwin Núñez.csv',
#     'David Raya Martin.csv': 'David Raya.csv',
#     'Deivid Washington de Souza Eugênio.csv': 'Deivid Washington.csv',
#     'Diego Carlos Santos Silva.csv': 'Diego Carlos.csv',
#     'Diogo Dalot Teixeira.csv': 'Diogo Dalot.csv',
#     'Diogo Teixeira da Silva.csv': 'Diogo Jota.csv',
#     'Douglas Luiz Soares de Paulo.csv': 'Douglas Luiz.csv',
#     'Ederson Santana de Moraes.csv': 'Ederson.csv',
#     'Edson Álvarez Velázquez.csv': 'Edson Álvarez.csv',
#     'Emerson Palmieri dos Santos.csv': 'Emerson.csv',
#     'Emiliano Buendía Stati.csv': 'Emiliano Buendía.csv',
#     'Emiliano Martínez Romero.csv': 'Emiliano Martinez.csv',
#     'Fábio Ferreira Vieira.csv': 'Fábio Vieira.csv',
#     'Gabriel dos Santos Magalhães.csv': 'Gabriel.csv',
#     'Gabriel Fernando de Jesus.csv': 'Gabriel Jesus.csv',
#     'Gabriel Martinelli Silva.csv': 'Gabriel Martinelli.csv',
#     'Gonçalo Manuel Ganchinho Guedes.csv': 'Gonçalo Guedes.csv',
#     'Hwang Hee-chan.csv': 'Hee-Chan Hwang.csv',
#     'Igor Julio dos Santos de Paulo.csv': 'Igor Julio.csv',
#     'Jefferson Lerma Solís.csv': 'Jefferson Lerma.csv',
#     'Joelinton Cássio Apolinário de Lira.csv': 'Joelinton.csv',
#     'Jorge Luiz Frello Filho.csv': 'Jorginho.csv',
#     'José Malheiro de Sá.csv': 'José Sá.csv',
#     'João Pedro Junqueira de Jesus.csv': 'João Pedro.csv',
#     'João Victor Gomes da Silva.csv': 'João Gomes.csv',
#     'João Palhinha Gonçalves.csv': 'João Palhinha.csv',
#     'Kepa Arrizabalaga.csv': 'Kepa.csv',
#     'Lucas Tolentino Coelho de Lima.csv': 'Lucas Paquetá.csv',
#     'Manuel Benson Hedilazio.csv': 'Benson Manuel.csv',
#     'Marc Cucurella Saseta.csv': 'Marc Cucurella.csv',
#     'Martin Ødegaard.csv': 'Martin Odegaard.csv',
#     'Matheus França de Oliveira.csv': 'Matheus França.csv',
#     'Matheus Luiz Nunes.csv': 'Matheus Nunes.csv',
#     'Matheus Santos Carneiro Da Cunha.csv': 'Matheus Cunha.csv',
#     'Miguel Almirón Rejala.csv': 'Miguel Almirón.csv',
#     'Murillo Santiago Costa dos Santos.csv': 'Murillo.csv',
#     'Norberto Bercique Gomes Betuncal.csv': 'Beto.csv',
#     'Norberto Murara Neto.csv': 'Neto.csv',
#     'Pedro Lomba Neto.csv': 'Pedro Neto.csv',
#     'Philippe Coutinho Correia.csv': 'Philippe Coutinho.csv',
#     'Richarlison de Andrade.csv': 'Richarlison.csv',
#     'Rúben Gato Alves Dias.csv': 'Rúben Dias.csv',
#     'Son Heung-min.csv': 'Son Heung-Min.csv',
#     'Thiago Alcántara do Nascimento.csv': 'Thiago Alcántara.csv',
#     'Thiago Emiliano da Silva.csv': 'Thiago Silva.csv',
#     'Tino Livramento.csv': 'Valentino Livramento.csv',
#     'Tomáš Souček.csv': 'Tomas Soucek.csv',
#     'Toti António Gomes.csv': 'Toti.csv',
#     'Vini de Souza Costa.csv': 'Vinicius Souza.csv',
#     'Willian Borges da Silva.csv': 'Willian.csv',
#     'Đorđe Petrović.csv': 'Djordje Petrovic.csv',
#     'Pervis Estupiñán.csv': 'Estupiñán.csv',

#     # --- New/Updated for the 24/25 Season ---
#     'Andreas Hoelgebaum Pereira.csv': 'Andreas Pereira.csv',
#     'Andrey Nascimento dos Santos.csv': 'Andrey Santos.csv',
#     'Carlos Alcaraz Durán.csv': 'Carlos Alcaraz.csv',
#     'Chadi Riad Dnanou.csv': 'Chadi Riad.csv',
#     'Cheick Doucouré.csv': 'Cheick Oumar Doucoure.csv',
#     'Dominic Solanke-Mitchell.csv': 'Dominic Solanke.csv',
#     'Emile Smith Rowe.csv': 'Emile Smith-Rowe.csv',
#     'Endo Wataru.csv': 'Wataru Endo.csv',
#     'Facundo Pellistri Rebollo.csv': 'Facundo Pellistri.csv',
#     'Fábio Freitas Gouveia Carvalho.csv': 'Fabio Carvalho.csv',
#     'Hamed Traorè.csv': 'Hamed Junior Traore.csv',
#     'Igor Thiago Nascimento Rodrigues.csv': 'Thiago.csv',
#     'Ismaïla Sarr.csv': 'Ismaila Sarr.csv',
#     'Jaden Philogene.csv': 'Jaden Philogene-Bidace.csv',
#     'Jeremy Sarmiento Morante.csv': 'Jeremy Sarmiento.csv',
#     'Joe Aribo.csv': 'Joe Ayodele-Aribo.csv',
#     'João Pedro Ferreira Silva.csv': 'Jota Silva.csv',
#     'Julián Araujo Zúñiga.csv': 'Julián Araujo.csv',
#     'Jérémy Doku.csv': 'Jéremy Doku.csv',
#     'Kaine Kesler-Hayden.csv': 'Kaine Hayden.csv',
#     'Luis Guilherme Lira dos Santos.csv': 'Luis Guilherme.csv',
#     'Mitoma Kaoru.csv': 'Kaoru Mitoma.csv',
#     'Moisés Caicedo Corozo.csv': 'Moisés Caicedo.csv',
#     'Nayef Aguerd.csv': 'Naif Aguerd.csv',
#     'Nélson Cabral Semedo.csv': 'Nélson Semedo.csv',
#     'Omari Giraud-Hutchinson.csv': 'Omari Hutchinson.csv',
#     'Pape Matar Sarr.csv': 'Pape Sarr.csv',
#     'Pedro Cardoso de Lima.csv': 'Pedro Lima.csv',
#     'Renato Palma Veiga.csv': 'Renato Veiga.csv',
#     'Ricardo Barbosa Pereira.csv': 'Ricardo Pereira.csv',
#     'Rodrigo Martins Gomes.csv': 'Rodrigo Gomes.csv',
#     'Rodrigo Muniz Carvalho.csv': 'Rodrigo Muniz.csv',
#     "Rodrigo 'Rodri' Hernandez.csv": 'Rodri.csv',
#     'Sam Szmodics.csv': 'Sammie Szmodics.csv',
#     "Sávio 'Savinho' Moreira de Oliveira.csv": 'Sávio.csv',
#     'Sugawara Yukinari.csv': 'Yukinari Sugawara.csv',
#     'Tomiyasu Takehiro.csv': 'Takehiro Tomiyasu.csv',
#     'Youssef Ramalho Chermiti.csv': 'Youssef Chermiti.csv',
#     'Ângelo Gabriel Borges Damaceno.csv': 'Angelo Gabriel.csv',
#     'Łukasz Fabiański.csv': 'Lukasz Fabianski.csv',

#     # --- Understat Special Character Fixes ---
#     'Jake O\'Brien.csv': 'Jake O&#039;Brien.csv',
#     'Nico O\'Reilly.csv': 'Nico O&#039;Reilly.csv',
# }
#     # 3. Create lookups
#     understat_simple_map = {normalize_name(x): x for x in understat_raw}

#     # 4. Supplemental Data: Load players_raw to handle "Web Name" mappings
#     web_name_map = {}
#     if os.path.exists(raw_players_path):
#         try:
#             df = pd.read_csv(raw_players_path)
#             # Link Full FPL Name (CSV format) to Web Name (CSV format)
#             for _, row in df.iterrows():
#                 full_fpl = f"{row['first_name']} {row['second_name']}.csv"
#                 web_name = f"{row['web_name']}.csv"
#                 web_name_map[full_fpl] = web_name
#         except Exception as e:
#             print(f"Note: Could not use players_raw.csv for mapping: {e}")

#     final_mapping = {}
#     unmapped = []

#     for fpl_file in fpl_raw:
#         # Step A: Exact Match
#         if fpl_file in understat_raw:
#             final_mapping[fpl_file] = fpl_file
#             continue

#         # Step B: Manual Overrides
#         if fpl_file in manual_overrides:
#             target = manual_overrides[fpl_file]
#             if target in understat_raw:
#                 final_mapping[fpl_file] = target
#                 continue

#         # Step C: Exact Normalized Match
#         fpl_norm = normalize_name(fpl_file)
#         if fpl_norm in understat_simple_map:
#             final_mapping[fpl_file] = understat_simple_map[fpl_norm]
#             continue

#         # Step D: Web Name Cross-Reference (from players_raw.csv)
#         if fpl_file in web_name_map:
#             web_target = web_name_map[fpl_file]
#             web_norm = normalize_name(web_target)
#             if web_norm in understat_simple_map:
#                 final_mapping[fpl_file] = understat_simple_map[web_norm]
#                 continue

#         # Step E: Substring Match (Case-insensitive)
#         found_substring = False
#         for u_norm, u_orig in understat_simple_map.items():
#             # Min length threshold to avoid 'Dan' matching 'Danny' incorrectly
#             if u_norm in fpl_norm and len(u_norm) > 4:
#                 final_mapping[fpl_file] = u_orig
#                 found_substring = True
#                 break
#         if found_substring: continue

#         # Step F: Fuzzy Match (Normalized)
#         matches = difflib.get_close_matches(fpl_norm, list(understat_simple_map.keys()), n=1, cutoff=0.85)
#         if matches:
#             final_mapping[fpl_file] = understat_simple_map[matches[0]]
#             continue

#         unmapped.append(fpl_file)

#     print(f"Successfully mapped {len(final_mapping)} players.")
#     if unmapped:
#         print(f"Could not map {len(unmapped)} players: {unmapped[:5]}...")
#         print(unmapped)

#     return pd.DataFrame({'fpl_name': final_mapping.keys(), 'understat_name': final_mapping.values()})

In [57]:
def save_fpl_players(fpl_subfolders, season):
    # Get player names
    all_rows = []

    for i in range(1, 39):
        gw_data = pd.read_csv(f'./data/vaastav/data/2023-24/gws/gw{i}.csv')
        gw_data["gw"] = i  # track gameweek
        subset = gw_data[['gw', 'element', 'name', 'team', 'position']]
        all_rows.append(subset)

    df = pd.concat(all_rows)
    gw_data = df.sort_values("gw").drop_duplicates(subset="element", keep="last")

    for folder in fpl_subfolders:
        # print(folder)
        # print(folder)
        player_name = folder.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        # print(clean_player_name)
        player = gw_data[gw_data['name'] == clean_player_name]
        if player.empty:
            # print(f"Player {clean_player_name} not found in gw_data")
            continue
        # print(player)

        player_df = pd.read_csv(str(folder+'/gw.csv'))

        # Find all rows where the 'round' is repeated and take the max total_points
        ## 1. Sort by 'round' (ascending) and 'total_points' (descending)
        ### This puts the row with the most points at the top of each round group
        df_sorted = player_df.sort_values(by=['round', 'total_points'], ascending=[True, False])

        ## 2. Drop duplicates based on 'round', keeping the 'first' row (the highest points)
        df_cleaned = df_sorted.drop_duplicates(subset=['round'], keep='first')

        ## 3. Sort back by round to keep chronological order
        df_cleaned = df_cleaned.sort_values('round')

        # fpl_id to df_cleaned
        fpl_id = folder.split('/')[6].split('_')[-1]
        df_cleaned['fpl_id'] = fpl_id
        df_cleaned['position'] = player['position']
        df_cleaned['team'] = player['team']
        df_cleaned['fpl_name'] = clean_player_name

        player_dir = './data/joint/'+str(season)+'/fpl/'

        # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        df_cleaned.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned 20'+ season +' fpl data')

# Understat Files
def save_under_players(understat_files, season):
    print('***********************', understat_files)
    for file_ in understat_files:
        player_name = file_.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        player_df = pd.read_csv(str(file_))
        player_dir = './data/joint/'+str(season)+'/understat/'

        # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        player_df.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned understat 20'+ season +' data')

def joint_players_info(fpl_player_folder_path, understat_player_folder_path, season):
    fpl_subfolders = [ f.path for f in os.scandir(fpl_player_folder_path) if f.is_dir() ]
    under_files = [ f.path for f in os.scandir(understat_player_folder_path) if f.is_file() ]

    save_fpl_players(fpl_subfolders, season)
    save_under_players(under_files, season)

    print('20'+ season +' fpl and understat data now in `joint` folder')

In [ ]:
def merge_fpl_understat_data(fpl_player_folder_path, understat_player_folder_path, season):
    joint_players_info(fpl_player_folder_path, understat_player_folder_path, season)
    joint_fpl_data_path = "./data/joint/"+ season + "/fpl/"
    joint_understat_path = "./data/joint/"+ season + "/understat/"

    # Get the player positions from the previous seasons
    play_merged = pd.read_csv(f'./data/vaastav/data/20{season}/gws/merged_gw.csv')
    unique_players = play_merged.drop_duplicates(subset=['name'])

    name_to_position_series = unique_players.set_index('name')['position']
    name_position_dict = name_to_position_series.to_dict()

    # understat_files = next(os.walk("./data/joint/"+ season + "/understat/"), (None, None, []))[2]  # [] if no file
    # fpl_files = next(os.walk( "./data/joint/"+ season +"/fpl"), (None, None, []))[2]  # [] if no file
    # fpl_understat_id_name = pd.read_csv('./fpl_understat_id_name.csv')

    # Clear out files that don't have both fpl and understat data
    sns_yr = '_'.join(season.split('-'))  # '2021-22' -> '2021'
    # fpl_understat_id_name =pd.read_csv('./data/vaastav/data/merged_names_'+ sns_yr +'.csv') # merged_names_23_24 pd.read_csv('./fpl_understat_id_name.csv')
    # fpl_understat_id_name.rename(columns={'FPL_Name': 'fpl_name', 'FPL_ID':'fpl_id' , 'Understat_ID': 'understat_id','Understat_Name': 'understat_name'}, inplace=True)

    # fpl_understat_id_name = map_names(season)
    mapped_names = map_names(season)
    print('-------------', mapped_names)

    fpl_raw = mapped_names['fpl_name'].values.tolist()
    under_raw = mapped_names['understat_name'].values.tolist()
    # understat_names = [file_.split('.')[0] for file_ in under_raw]
    # fpl_names = [file_.split('.')[0] for file_ in fpl_raw]

    for row in mapped_names.itertuples():
        fpl_link = row.fpl_name
        understat_link = row.understat_name

        fpl_name = fpl_link.split('.')[0]
        understat_name = understat_link.split('.')[0]

        try:
            fpl_player_data = pd.read_csv(joint_fpl_data_path + fpl_link)
            understat_player_data = pd.read_csv(joint_understat_path + understat_link)

            # Change 'kickoff_time' column name to 'date
            fpl_player_data = fpl_player_data.rename(columns={'kickoff_time': 'date'})

            # change the formats: From 2021-10-03T13:00:00Z to 2021-10-03
            fpl_player_data.date = fpl_player_data.date.apply(lambda x: x.split('T')[0])

            sns = f"20{season.split('-')[0]}"
            understat_filtered = understat_player_data[understat_player_data['season'] == int(sns)]


            # Marge fpl_player_data with understat_player_data if the dates match
            player_data_merged = fpl_player_data.merge(understat_filtered, on="date")

            def set_player_team(row):
                # add fpl_id
                # row['fpl_id'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_id'].values[0]
                row['understat_id'] = row['id']   #fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['understat_id'].values[0]

                # # add player_team
                h_team = player_data_merged.iloc[0]['h_team']
                a_team = player_data_merged.iloc[0]['a_team']
                # opponent_team = fpl_player.iloc[0]['opponent_team']
                was_home = player_data_merged.iloc[0]['was_home']
                row['player_team'] = h_team if was_home else a_team

                # add FPL_Name
                row['FPL_Name'] = fpl_name
                # add Understat_Name
                row['Understat_Name'] = understat_name

                # add fpl position
                row['position'] = name_position_dict[fpl_name]

                return row

            player_data_merged = player_data_merged.apply(set_player_team, axis=1)
            print('............... Saving data')
            if(player_data_merged.shape[0]):
                merged_dir = './data/joint/'+ season +'/merged/'
                if not os.path.exists(merged_dir):
                    os.makedirs(merged_dir)

                player_data_merged.to_csv(merged_dir+ fpl_link, index_label=False )

        except pd.errors.EmptyDataError:
            print(f"⚠️ Warning: The file '{joint_fpl_data_path + fpl_link}' is empty. Skipping.")
            # You can choose to create an empty DataFrame or just continue
            print(f"Error reading files for {fpl_link} or {understat_link}. Skipping...")
            continue






    # fpl_names = fpl_understat_id_name['fpl_name'].values.tolist()
    # under_names = fpl_understat_id_name['understat_name'].values.tolist()


    # understat_files = next(os.walk("./data/joint/"+ season + "/understat/"), (None, None, []))[2]  # [] if no file
    # fpl_files = next(os.walk( "./data/joint/"+ season +"/fpl"), (None, None, []))[2]  # [] if no file


    # fpl_files_ = [f.split('.')[0] for f in fpl_files]
    # understat_files_ = [f.split('.')[0] for f in understat_files]

    # # drop fple_files not in fpl_names
    # fpl_file_names = [file_ for file_ in fpl_files_ if file_ in fpl_names]
    # under_file_name = [file_ for file_ in understat_files_ if file_ in under_names]


    # # put back to fpl_files
    # fpl_files =[name+'.csv' for name in fpl_file_names]
    # understat_files = [name+'.csv' for name in under_file_name]

    # understat_names = [file_.split('.')[0] for file_ in understat_files]
    # fpl_file_names = [file_.split('.')[0] for file_ in fpl_files]

    # for name in fpl_file_names:
    #     fpl_player = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == name]


    #     if fpl_player['fpl_name'].values.size > 0:
    #         fpl_player_name = fpl_player['fpl_name'].values[0]
    #         understat_player_name = fpl_player['understat_name'].values[0]

    #         try:
    #             fpl_player_data = pd.read_csv(joint_fpl_data_path + fpl_player_name+ '.csv')
    #             understat_player_data = pd.read_csv(joint_understat_path + understat_player_name + '.csv')

    #         except pd.errors.EmptyDataError:
    #             print(f"⚠️ Warning: The file '{joint_fpl_data_path + fpl_player_name+ '.csv'}' is empty. Skipping.")
    #             # You can choose to create an empty DataFrame or just continue
    #             print(f"Error reading files for {fpl_player_name} or {understat_player_name}. Skipping...")
    #             continue


    #         # print(joint_fpl_data_path + fpl_player_name+ '.csv')
    #         # Change 'kickoff_time' column name to 'date
    #         fpl_player_data = fpl_player_data.rename(columns={'kickoff_time': 'date'})
    #         # change the formats: From 2021-10-03T13:00:00Z to 2021-10-03
    #         fpl_player_data.date = fpl_player_data.date.apply(lambda x: x.split('T')[0])

    #         # Dates are of the form 2021-10-03T13:00:00Z
    #         fpl_dates_min = fpl_player_data['date'].min()
    #         fpl_dates_max = fpl_player_data['date'].max()

    #         # # Filter out player info not in the range of dates we are dealing with
    #         # understat_filtered = understat_player_data[(pd.to_datetime(understat_player_data['date']) >= pd.to_datetime(fpl_dates_min))
    #         #                                             & (pd.to_datetime(understat_player_data['date']) <= pd.to_datetime(fpl_dates_max) )]

    #         sns = f"20{season.split('-')[0]}"

    #         understat_filtered = understat_player_data[understat_player_data['season'] == int(sns)]

    #         # Marge fpl_player_data with understat_player_data if the dates match
    #         player_data_merged = fpl_player_data.merge(understat_filtered, on="date")

    #         # Add player team
    #         def set_player_team(row):
    #             # add fpl_id
    #             # row['fpl_id'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_id'].values[0]
    #             row['understat_id'] = row['id']   #fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['understat_id'].values[0]
    #             # # add player_team
    #             if 'team' not in fpl_understat_id_name.columns:
    #                 # get the player team from understat data
    #                 # under_player = pd.read_csv(f'./vaastav/data/{season}/understat/Aaron_Cresswell_534.csv')
    #                 # fpl_player = pd.read_csv(f'./vaastav/data/{season}/players/Aaron_Cresswell_517/gw.csv')
    #                 # teams = pd.read_csv(f'./vaastav/data/{season}/teams.csv')

    #                 # Get player team by comparing opponent_team in fpl gw data with teams id in teams data
    #                 h_team = player_data_merged.iloc[0]['h_team']
    #                 a_team = player_data_merged.iloc[0]['a_team']
    #                 # opponent_team = fpl_player.iloc[0]['opponent_team']
    #                 was_home = player_data_merged.iloc[0]['was_home']
    #                 row['player_team'] = h_team if was_home else a_team
    #             else:
    #                 row['player_team'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['team'].values[0]

    #             # add FPL_Name
    #             row['FPL_Name'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_name'].values[0]
    #             # add Understat_Name
    #             row['Understat_Name'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['understat_name'].values[0]

    #             # add fpl position
    #             if 'fpl_position' not in fpl_understat_id_name.columns:
    #                row['position'] = name_position_dict[fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_name'].values[0]]
    #             else:
    #                 row['position'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_position'].values[0]
    #             return row
    #         player_data_merged = player_data_merged.apply(set_player_team, axis=1)

    #         #     return row['h_team'] if row['was_home'] else row['a_team']

    #         # player_data_merged['player_team'] = player_data_merged.apply(set_player_team, axis=1)

    #         if(player_data_merged.shape[0]):
    #             merged_dir = './data/joint/'+ season +'/merged/'
    #             if not os.path.exists(merged_dir):
    #                 os.makedirs(merged_dir)

    #             player_data_merged.to_csv(merged_dir+ fpl_player_name +'.csv', index_label=False )

In [41]:
def add_difficulty(season):
    print('====> Starting to add difficulty features to 20'+season)
    merged = './data/joint/' + season + '/merged/'

    player_names = next(os.walk((merged), (None, None, [])))[2]
    fixtures = pd.read_csv('./data/vaastav/data/20' + season + '/fixtures.csv')

    # Loop over each player file in player_names
    for name in player_names:
        # Load player data
        player = pd.read_csv('./data/joint/' + season + '/merged/' + name)

        # Function to get the difficulty and was_home columns based on the fixture
        def get_fixture_info(row):
            # Filter the relevant fixture
            fixture = fixtures[fixtures['id'] == row['fixture']]
            if not fixture.empty:
                fixture = fixture.iloc[0]  # Get the first (and only) match

                # Get the team difficulties
                team_h_difficulty = fixture['team_h_difficulty']
                team_a_difficulty = fixture['team_a_difficulty']

                event = fixture['event']

                return pd.Series([team_h_difficulty, team_a_difficulty, event])
            else:
                # Return NaN if no matching fixture found
                return pd.Series([None, None, None])

        # Apply the function to each row of player
        player[['team_h_difficulty', 'team_a_difficulty', 'event']] = player.apply(get_fixture_info, axis=1)
        # Update the value to prices
        # player['value'] = player['value']/10

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/' + season + '/merged_extras/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + name, index=False)

In [42]:
def add_xP(season, gwk=None):
    print('================> starting to add xp for season 20'+season)
    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        player_data = None
        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            players = data['elements']  # Extract the list of players

            player_data = [
                {"id": player["id"],  "name": f"{player['first_name']} {player['second_name']}", "player_team": player["team"], "price": player["now_cost"] / 10, "position": player["element_type"]}
                for player in players
            ]

        else:
            print('Failed to retrieve data')

        fpl_players = pd.DataFrame(player_data)

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras', [None], [None],[]))[2]
    # players_paths
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras/'+ path)
        # print(path)
        # print(player['fpl_name'][0])
        merged = pd.read_csv('./data/vaastav/data/20'+ season +'/gws/merged_gw.csv', low_memory=False)

        player = player.drop(['position'], axis=1)
        merged_player = pd.merge(player, merged[['element', 'fixture', 'xP','position']], on=['element', 'fixture'], how='left')
        # drop the duplicates
        merged_player = merged_player.drop_duplicates(subset=['date'])
        # print(path ,player.shape, merged.shape, merged_player.shape)

        if gwk:
            merged_player = merged_player.reindex(merged_player.index.tolist()  + list([merged_player.index[-1]+1]))


            player_id = int(merged_player.iloc[-2, merged_player.columns.get_loc('element')])
            fpl_data = fpl_players[fpl_players['id'] == player_id]

            merged_player.iloc[-1, merged_player.columns.get_loc('event')] = gwk
            merged_player.iloc[-1, merged_player.columns.get_loc('value')] = fpl_data['price'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('position')] = fpl_data['position'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('fpl_id')] = player['element'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('fpl_name')] = player['fpl_name'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('understat_id')] = player['understat_id'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('player_team')] = fpl_data['player_team'].values[0]

        merged_player['pts_bps'] = merged_player['total_points'] - merged_player['bonus']

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ season +'/merged_extras_xP/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        merged_player.to_csv(new_col_dir + path, index=False)


In [43]:
def add_rolling_avgs(season):
    print('================> starting to roll by 2 for season 20'+season)
    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_xP', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_xP/'+ path,sep=',', skipinitialspace=True)

        features = ['clean_sheets', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'goals_conceded', 'goals_scored', 'ict_index',
                        'influence', 'creativity', 'threat', 'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'yellow_cards', 'saves', 'starts',
                        'team_a_score', 'team_h_score', 'pts_bps', 'goals', 'shots', 'xG', 'xA', 'assists_x', 'assists_y', 'key_passes', 'npg', 'npxG', 'xGChain',  'xGBuildup',  'xP']

        for i in [1,3,5]:
            # Compute rolling 5-game averages, shifted by 1 (efficient method)
            # rolling_means = round(player[features].rolling(window=i).mean().shift(1), 2)

            # Compute rolling 5-game sum, shifted by 1
            rolling_sum = player[features].rolling(window=i, min_periods=1).sum().shift(1)

            # Divide the sum by the fixed window size 'i'
            rolling_means = round(rolling_sum / i, 2)
            rolling_means.columns = [f"{col}_{i}" for col in rolling_means.columns]  # Rename new columns
            # print(i)
            # Concatenate new rolling mean columns efficiently
            player = pd.concat([player, round(rolling_means, 2)], axis=1)
            # print(f"Rolling {i} added for {path}", player)
            # Defragment memory
            player = player.copy()

        # divide value by 10 to get player price
        player['value'] = player['value'] / 10
        # Save the updated DataFrame with the new columns
        # print(player[['minutes', 'minutes_3','minutes_4', 'minutes_5']].head(10))
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

    print('<<<<================ Done rolling season 20'+season)

In [44]:
def owenership_change(season, gwk=None):
    print('================> starting to add ownership change for season 20'+season)
    players = []
    transfers_df = None
    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        # print(response.status_code)

        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            # print("data: ", data)
            players = data['elements']  # Extract the list of players

            # print(players)
            transfers = [{
                "id": player["id"],
                'ict_index': player['ict_index'],
                'influence': player['influence'],
                'creativity': player['creativity'],
                'threat': player['threat'],
                "transfers_in": player["transfers_in_event"],
                "transfers_out": player["transfers_out_event"]
            } for player in players]
            transfers_df = pd.DataFrame(transfers)
        else:
            print('Failed to retrieve data')

        # transfers_df = pd.DataFrame(transfers)
        # print(transfers_df[['ict_index', 'influence', 'creativity', 'threat']])

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_rolled', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_rolled/'+ path,sep=',', skipinitialspace=True)
        player['ownership_change'] = player['selected'].diff().fillna(0)

        def ownership_change(row):
            net_transfers = row['transfers_in'] - row['transfers_out']
            total_transfers = row['transfers_in'] + row['transfers_out']
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            return net_transfers_pct

        player['percenatge_net_transfers'] = player.apply(ownership_change, axis=1)

        if gwk:
            # print('------------>',len(transfers_df.columns))
            player_id = int(player.iloc[-2, player.columns.get_loc('element')])
            transfer_data = transfers_df[transfers_df['id'] == player_id]
            net_transfers = transfer_data['transfers_in'].values[0] - transfer_data['transfers_out'].values[0]
            total_transfers = transfer_data['transfers_in'].values[0] + transfer_data['transfers_out'].values[0]
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            player.iloc[-1, player.columns.get_loc('percenatge_net_transfers')] = net_transfers_pct
            # print(transfer_data['ict_index'].values[0])
            player['ict_index'] = transfer_data['ict_index'].values[0]
            player['influence'] = transfer_data['influence'].values[0]
            player['creativity'] = transfer_data['creativity'].values[0]
            player['threat'] = transfer_data['threat'].values[0]

        # Save the updated DataFrame with the new columns
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled_net_transfers/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)


In [45]:
def odds(sns, nxt_gw=0):
    print('================> starting adding odds for sns 20'+sns)

    teams_25_26 = [
        {"name":"Arsenal", "id":"3","shortName":"Arsenal","abbr":"ARS"},
        {"name":"Aston Villa", "id":"7","shortName":"Aston Villa","abbr":"AVL"},
        {"name":"Bournemouth", "id":"91","shortName":"Bournemouth","abbr":"BOU"},
        {"name":"Brentford", "id":"94","shortName":"Brentford","abbr":"BRE"},
        {"name" :"Brighton and Hove Albion","id":"36","shortName":"Brighton","abbr":"BHA"},
        {"name":"Burnley","id":"90","shortName":"Burnley","abbr":"BUR"},
        {"name":"Chelsea","id":"8","shortName":"Chelsea","abbr":"CHE"},
        {"name":"Crystal Palace","id":"31","shortName":"Crystal Palace","abbr":"CRY"},
        {"name":"Everton","id":"11","shortName":"Everton","abbr":"EVE"},
        {"name":"Fulham","id":"54","shortName":"Fulham","abbr":"FUL"},
        {"name":"Leeds United","id":"2","shortName":"Leeds","abbr":"LEE"},
        {"name":"Liverpool","id":"14","shortName":"Liverpool","abbr":"LIV"},
        {"name":"Manchester City","id":"43","shortName":"Man City","abbr":"MCI"},
        {"name":"Manchester United","id":"1","shortName":"Man Utd","abbr":"MUN"},
        {"name":"Newcastle United","id":"4","shortName":"Newcastle","abbr":"NEW"},
        {"name":"Nottingham Forest","id":"17","shortName":"Nott'm Forest","abbr":"NFO"},
        {"name":"Sunderland","id":"56","shortName":"Sunderland","abbr":"SUN"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Spurs","abbr":"TOT"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Tottenham","abbr":"TOT"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Tottenham Hotspur","abbr":"TOT"},
        {"name":"West Ham United","id":"21","shortName":"West Ham","abbr":"WHU"},
        {"name":"Wolverhampton Wanderers","id":"39","shortName":"Wolves","abbr":"WOL"}]

    # create lookup dictionary
    team_name_to_id_25_26 = {team['shortName']: team['name'] for team in teams_25_26}

    # Load the data data once to avoid redundant file reads
    data = pd.read_csv('./data/odds/E0 '+ sns +'.csv')

    data = data.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

    # Update the odds-data names of the team to match the fpl names
    # use teams dict to update team names
    def update_name(row):
        if row['h_team'] in team_name_to_id_25_26:
            row['h_team'] = team_name_to_id_25_26[row['h_team']]
        if row['a_team'] in team_name_to_id_25_26:
            row['a_team'] = team_name_to_id_25_26[row['a_team']]
        return row

    data = data.apply(update_name, axis=1)

    players_paths = next(os.walk('./data/joint/'+ sns +'/merged_extras_rolled_net_transfers', [None], [None],[]))[2]
    # print(len(players_paths), 'players to add odds to')
    for path in players_paths:
        rolled = pd.read_csv('./data/joint/'+ sns +'/merged_extras_rolled_net_transfers/'+ path,sep=',', skipinitialspace=True)

        def update_player_team(row):
            if row['player_team'] in team_name_to_id_25_26:
                row['player_team'] = team_name_to_id_25_26[row['player_team']]
            if row['opponent_team'] in team_name_to_id_25_26:
                row['opponent_team'] = team_name_to_id_25_26[row['opponent_team']]
            if row['h_team'] in team_name_to_id_25_26:
                row['h_team'] = team_name_to_id_25_26[row['h_team']]
            if row['a_team'] in team_name_to_id_25_26:
                row['a_team'] = team_name_to_id_25_26[row['a_team']]
            return row
        rolled = rolled.apply(update_player_team, axis=1)

        def add_odds(row, data):
            # Filter the data DataFrame for the matching teams
            # print(data)
            match = data[(data['h_team'] == row['h_team']) & (data['a_team'] == row['a_team'])]
            # Check if a match is found
            if not match.empty:
                # print(row['value'])

                # Extract the relevant data values
                # Convert the data to probabilities
                odds_ = match.iloc[0]

                WHH = round(1/odds_['B365H'], 5)
                WHD = round(1/odds_['B365D'], 5)
                WHA = round(1/odds_['B365A'], 5)

                # Normalize the probabilities (to make the probabilities sum to 100%)
                WH_sum = WHH + WHD + WHA
                WHH_ = round(WHH/WH_sum, 3)
                WHD_ = round(WHD/WH_sum, 3)
                WHA_ = round(WHA/WH_sum, 3)

                # if row['minutes'] == 0:
                #     xG_90 = 0
                #     xA_90 = 0
                # else:
                #     xG_90 = (row['xG']/row['minutes'])*90
                #     xA_90 = (row['xA']/row['minutes'])*90
                # pts_bps = row['total_points'] - row['bonus']
                return pd.Series([WHH_, WHD_, WHA_])
            else:
                # Return NaN for rows with no match
                return pd.Series([None,None, None])

        # # Apply the function to the 'rolled' DataFrame
        # rolled[['pts_bps', 'whh', 'whd', 'wha']] = rolled.apply(add_odds, axis=1, data=data)

        # 1. Run .apply() and store the results in a NEW DataFrame
        new_columns = rolled.apply(add_odds, axis=1, data=data)

        # Optional but good practice: ensure the new columns have the correct names
        # (Your .apply() function might already return a Series with names, but this is a safe way to be sure)
        new_columns.columns = ['whh', 'whd', 'wha'] # ['pts_bps', 'whh', 'whd', 'wha']

        # 2. Join the original DataFrame with the new columns all at once
        rolled = pd.concat([rolled, new_columns], axis=1)

        if nxt_gw:
            odds_nxt = pd.read_csv(f'./data/odds/odds_{nxt_gw}.csv')
            odds_nxt = odds_nxt.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

            def update_name(row):
                if row['h_team'] in team_name_to_id_25_26:
                    row['h_team'] = team_name_to_id_25_26[row['h_team']]
                if row['a_team'] in team_name_to_id_25_26:
                    row['a_team'] = team_name_to_id_25_26[row['a_team']]
                return row
            odds_nxt = odds_nxt.apply(update_name, axis=1)

            # print(odds_nxt["a_team"].unique(), odds_nxt['h_team'].unique(), rolled['player_team'].unique())
            opp_team = rolled['opponent_team']
            player_team = rolled['player_team'].loc[0]
            # print(row)

            team_odds_nxt = odds_nxt[(odds_nxt['h_team'] == player_team) | (odds_nxt['a_team']==player_team)]
            # print('------------------->', team_odds_nxt)
            if(team_odds_nxt.empty):
                print(f'No odds found for {player_team} in gw{nxt_gw}, skipping...')
                continue


            h_team = team_odds_nxt.loc[:, 'h_team'].values[0]
            a_team = team_odds_nxt.loc[:, 'a_team'].values[0]
            whh = team_odds_nxt.loc[:,'WHH'].values[0]
            whd = team_odds_nxt.loc[:,'WHD'].values[0]
            wha = team_odds_nxt.loc[:,'WHA'].values[0]
            h_fdr = team_odds_nxt.loc[:,'h_fdr'].values[0]
            a_fdr = team_odds_nxt.loc[:,'a_fdr'].values[0]
            was_home = True if h_team == player_team else False

            rolled.iloc[-1, rolled.columns.get_loc('whh')] = whh
            rolled.iloc[-1, rolled.columns.get_loc('whd')] = whd
            rolled.iloc[-1, rolled.columns.get_loc('wha')] = wha
            rolled.iloc[-1, rolled.columns.get_loc('was_home')] = was_home
            rolled.iloc[-1, rolled.columns.get_loc('h_team')] = h_team
            rolled.iloc[-1, rolled.columns.get_loc('a_team')] = a_team
            rolled.iloc[-1, rolled.columns.get_loc('team_h_difficulty')] = h_fdr
            rolled.iloc[-1, rolled.columns.get_loc('team_a_difficulty')] = a_fdr

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ sns +'/merged_extras_odds/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        rolled.to_csv(new_col_dir + path, index=False)

In [46]:
def merge_files(season):
    print('starting to merge files for 20'+ season)
    paths = next(os.walk('./data/joint/'+ season +'/merged_extras_odds', [None], [None],[]))[2]
    print(len(paths))
    files_list = [pd.read_csv('./data/joint/'+ season +'/merged_extras_odds/' + path)  for  path in paths ]
    merged_files = pd.concat(files_list)

    # print(merged_files['fpl_id'])
    # Save the new DataFrame
    new_col_dir = './data/joint/'+ season +'/'

    merged_files.to_csv(new_col_dir  +'merged_player_data.csv', index=False)